# Preprocesamiento para el modelo

Ingeniería de características sobre las variables seleccionadas en el análisis exploratorio (`MODEL_FEATURES` en `pipeline_config.yaml`). Se aplican, en este orden:

1. Imputación de variables categóricas y numéricas.
2. Codificación de variables categóricas.
3. Tratamiento de outliers.
4. Transformación de variables continuas sesgadas.
5. Escalado de variables.

Todas las transformaciones se hacen con **feature-engine** y se integran en un **pipeline de sklearn**, que se ajusta solo con el conjunto de entrenamiento. El resultado son los datasets transformados en `data/processed/`, que serán el input del notebook de modelos.

## Librerías

In [1]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.encoding import OrdinalEncoder, RareLabelEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Configuración

Rutas y parámetros del preprocesamiento. Las rutas se pueden cambiar con variables de entorno, si cambia la ubicación de las fuentes de información o de las salidas.

In [2]:
# Rutas: modificar estas variables si cambia la ubicación de las fuentes o de las salidas
PROJ_DIR = Path(os.getenv("PROJ_DIR", ".."))
CONFIG_FILE = Path(os.getenv("PIPELINE_CONFIG", PROJ_DIR / "pipeline_config.yaml"))
RAW_DATA_DIR = Path(os.getenv("RAW_DATA_DIR", PROJ_DIR / "data" / "raw"))
PROCESSED_DATA_DIR = Path(os.getenv("PROCESSED_DATA_DIR", PROJ_DIR / "data" / "processed"))
MODELS_DIR = Path(os.getenv("MODELS_DIR", PROJ_DIR / "models"))

TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
PIPELINE_FILE = "preprocesamiento_pipeline.joblib"

TARGET = "SalePrice"
ID_COL = "Id"

# Parámetros del preprocesamiento
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_VALORES_DISCRETA = 15
NUMERICAS_COMO_CATEGORICAS = ["MSSubClass"]  # códigos sin orden
TOL_RARAS = 0.01
FOLD_IQR = 1.5
UMBRAL_SESGO = 0.75

## Carga de datos y features

Se cargan los datasets de train y test, usando `Id` como índice para conservar el identificador de cada casa en las salidas, y la lista de features del archivo de configuración.

In [3]:
train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE, index_col=ID_COL)
test = pd.read_csv(RAW_DATA_DIR / TEST_FILE, index_col=ID_COL)

with open(CONFIG_FILE, encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

MODEL_FEATURES = config["MODEL_FEATURES"]

print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"MODEL_FEATURES ({len(MODEL_FEATURES)}): {MODEL_FEATURES}")

train: (1460, 80)
test: (1459, 79)
MODEL_FEATURES (30): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage', 'OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'MSSubClass', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath', 'Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'Foundation', 'GarageType', 'BsmtFinType1', 'HeatingQC']


## Tipos de variables

Las features se agrupan con el mismo criterio del análisis exploratorio:

- **Categóricas:** las de texto y `MSSubClass`, que es numérica pero representa un código de tipo de vivienda sin orden.
- **Discretas:** numéricas con 15 valores distintos o menos.
- **Continuas:** el resto de las numéricas.

In [4]:
categoricas = [
    col
    for col in MODEL_FEATURES
    if not pd.api.types.is_numeric_dtype(train[col]) or col in NUMERICAS_COMO_CATEGORICAS
]
numericas = [col for col in MODEL_FEATURES if col not in categoricas]
discretas = [col for col in numericas if train[col].nunique() <= MAX_VALORES_DISCRETA]
continuas = [col for col in numericas if col not in discretas]

print(f"categoricas ({len(categoricas)}): {categoricas}")
print(f"discretas ({len(discretas)}): {discretas}")
print(f"continuas ({len(continuas)}): {continuas}")

categoricas (11): ['MSSubClass', 'Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'Foundation', 'GarageType', 'BsmtFinType1', 'HeatingQC']
discretas (9): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath']
continuas (10): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']


## División train / validación

`test.csv` no trae `SalePrice`, por lo que no sirve para evaluar modelos. Se separa el 20% de `train.csv` como validación; el pipeline se ajusta solo con el 80% restante y luego se aplica a validación y a test, para no filtrar información.

El target se transforma con `log(1 + SalePrice)` para corregir su sesgo a la derecha; en el notebook de modelos las predicciones se regresan a la escala original con `np.expm1`.

In [5]:
X = train[MODEL_FEATURES]
y = np.log1p(train[TARGET])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_test = test[MODEL_FEATURES]

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

X_train: (1168, 30)
X_val: (292, 30)
X_test: (1459, 30)


## 1. Imputación de variables categóricas y numéricas

- **Categóricas:** se imputan con la moda (categoría más frecuente) con `CategoricalImputer`.
- **Numéricas (continuas y discretas):** se imputan con la mediana con `MeanMedianImputer`, que es robusta al sesgo.

Se aprende un valor para todas las variables, aunque no tengan vacíos en train, porque test sí tiene vacíos en algunas de ellas (por ejemplo `GarageArea`, `TotalBsmtSF` y `KitchenQual`).

In [6]:
imputador_categoricas = CategoricalImputer(
    imputation_method="frequent", variables=categoricas, ignore_format=True
)
imputador_numericas = MeanMedianImputer(imputation_method="median", variables=numericas)

X_imputado = imputador_categoricas.fit_transform(X_train)
X_imputado = imputador_numericas.fit_transform(X_imputado)

print(f"Vacíos antes: {X_train.isna().sum().sum()}")
print(f"Vacíos después: {X_imputado.isna().sum().sum()}")

Vacíos antes: 1018
Vacíos después: 0


## 2. Codificación de variables categóricas

1. **Agrupación de categorías raras:** con `RareLabelEncoder`, las categorías presentes en menos del 1% de las casas se agrupan en `"Rare"`. Así, las categorías que no aparecen en train (por ejemplo `MSSubClass = 150`, que solo está en test) también se convierten en `"Rare"` al transformar.
2. **Codificación ordinal ordenada por target:** con `OrdinalEncoder(encoding_method="ordered")`, cada categoría se reemplaza por un número según la media de `SalePrice` de su grupo (0 = la categoría con la media más baja).
3. **Respaldo para categorías no vistas:** si una variable no tuvo categorías raras en el entrenamiento, `"Rare"` no tiene código; con `unseen="encode"` esos casos se codifican como -1, por debajo de la categoría con la media más baja (por ejemplo `HeatingQC = "Po"` en algún fold de la validación cruzada).

In [ ]:
agrupador_raras = RareLabelEncoder(
    tol=TOL_RARAS,
    n_categories=1,
    replace_with="Rare",
    variables=categoricas,
    ignore_format=True,
)
codificador = OrdinalEncoder(
    encoding_method="ordered", variables=categoricas, ignore_format=True, unseen="encode"
)

X_codificado = agrupador_raras.fit_transform(X_imputado)
X_codificado = codificador.fit_transform(X_codificado, y_train)

# Ejemplo de la codificación aprendida
pd.Series(codificador.encoder_dict_["ExterQual"], name="ExterQual")

## 3. Tratamiento de outliers

Los outliers de las variables continuas se limitan (*capping*) con `Winsorizer` usando el rango intercuartil: los valores por debajo de Q1 − 1.5·IQR o por encima de Q3 + 1.5·IQR se reemplazan por esos límites. No se eliminan filas, porque el pipeline también se aplica a test.

In [8]:
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=FOLD_IQR, variables=continuas)

X_sin_outliers = winsorizer.fit_transform(X_codificado)

# Número de valores limitados por variable
(X_codificado[continuas] != X_sin_outliers[continuas]).sum().rename("valores_limitados")

GrLivArea       23
GarageArea      18
TotalBsmtSF     47
1stFlrSF        13
YearBuilt        5
YearRemodAdd     0
GarageYrBlt      1
MasVnrArea      77
BsmtFinSF1       5
LotFrontage     85
Name: valores_limitados, dtype: int64

## 4. Transformación de variables continuas

Se aplica `YeoJohnsonTransformer` solo a las variables continuas con |sesgo| > 0.75, calculado en train después del tratamiento de outliers. Yeo-Johnson busca la potencia (λ) que deja cada variable lo más simétrica posible y, a diferencia de Box-Cox, acepta ceros.

In [9]:
sesgo = X_sin_outliers[continuas].skew()
continuas_sesgadas = sesgo[sesgo.abs() > UMBRAL_SESGO].index.tolist()

yeo_johnson = YeoJohnsonTransformer(variables=continuas_sesgadas)

X_transformado = yeo_johnson.fit_transform(X_sin_outliers)

pd.DataFrame(
    {
        "sesgo_antes": sesgo[continuas_sesgadas],
        "sesgo_despues": X_transformado[continuas_sesgadas].skew(),
        "lambda": pd.Series(yeo_johnson.lambda_dict_),
    }
).round(3)

,sesgo_antes,sesgo_despues,lambda
MasVnrArea,1.275,0.375,-0.218


## 5. Escalado de variables

Todas las features se estandarizan con `StandardScaler` (media 0 y desviación estándar 1), integrado al pipeline con `SklearnTransformerWrapper` de feature-engine para conservar el formato de DataFrame.

In [10]:
escalador = SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)

X_escalado = escalador.fit_transform(X_transformado)

X_escalado.agg(["mean", "std"]).round(2)

,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,...,Neighborhood,ExterQual,BsmtQual,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC
mean,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,-0.0,0.0,...,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Pipeline de preprocesamiento

Se integran todos los pasos anteriores en un pipeline de sklearn, se ajusta con train y se aplica a train, validación y test. Se verifica que el resultado coincida con el de los pasos individuales.

In [11]:
pipeline_preprocesamiento = Pipeline(
    steps=[
        ("imputacion_categoricas", imputador_categoricas),
        ("imputacion_numericas", imputador_numericas),
        ("agrupacion_raras", agrupador_raras),
        ("codificacion", codificador),
        ("outliers", winsorizer),
        ("transformacion", yeo_johnson),
        ("escalado", escalador),
    ]
)

pipeline_preprocesamiento.fit(X_train, y_train)

X_train_procesado = pipeline_preprocesamiento.transform(X_train)
X_val_procesado = pipeline_preprocesamiento.transform(X_val)
X_test_procesado = pipeline_preprocesamiento.transform(X_test)

print(f"Coincide con los pasos individuales: {np.allclose(X_train_procesado, X_escalado)}")
print(f"Vacíos en train / validación / test: "
      f"{X_train_procesado.isna().sum().sum()} / "
      f"{X_val_procesado.isna().sum().sum()} / "
      f"{X_test_procesado.isna().sum().sum()}")

pipeline_preprocesamiento

Coincide con los pasos individuales: True
Vacíos en train / validación / test: 0 / 0 / 0


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputacion_categoricas', ...), ('imputacion_numericas', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,imputation_method,'frequent'
,variables,"['MSSubClass', 'Neighborhood', ...]"
,ignore_format,True
,fill_value,'Missing'


## Guardar los datasets transformados

Se guardan en `data/processed/` como CSV, con `Id` como índice. Train y validación incluyen el target `SalePrice` en escala logarítmica; test no lo trae.

In [12]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

salidas = {
    "train": X_train_procesado.assign(**{TARGET: y_train}),
    "validation": X_val_procesado.assign(**{TARGET: y_val}),
    "test": X_test_procesado,
}

for nombre, datos in salidas.items():
    ruta = PROCESSED_DATA_DIR / f"{nombre}.csv"
    datos.to_csv(ruta)
    print(f"{nombre}: {datos.shape} -> {ruta.resolve()}")

train: (1168, 31) -> C:\Users\Usuario\OneDrive\Desktop\2026\Maestria\Cuarto Trimestre\PRODUCT DEVELOPMENT\Clase 2\clase_2\data\processed\train.csv
validation: (292, 31) -> C:\Users\Usuario\OneDrive\Desktop\2026\Maestria\Cuarto Trimestre\PRODUCT DEVELOPMENT\Clase 2\clase_2\data\processed\validation.csv
test: (1459, 30) -> C:\Users\Usuario\OneDrive\Desktop\2026\Maestria\Cuarto Trimestre\PRODUCT DEVELOPMENT\Clase 2\clase_2\data\processed\test.csv


## Guardar el pipeline y la configuración

El pipeline ajustado se guarda en `models/` con joblib, para aplicarlo a datos nuevos. En `pipeline_config.yaml` se agregan los parámetros del preprocesamiento y las rutas de las salidas, conservando `MODEL_FEATURES`.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ruta_pipeline = MODELS_DIR / PIPELINE_FILE
joblib.dump(pipeline_preprocesamiento, ruta_pipeline)


def ruta_relativa(ruta):
    """Ruta relativa a la raíz del proyecto, para guardarla en la configuración."""
    return Path(os.path.relpath(ruta, PROJ_DIR)).as_posix()


config.update(
    {
        "TARGET": TARGET,
        "TARGET_TRANSFORMATION": "log1p",
        "ID_COLUMN": ID_COL,
        "VARIABLES": {
            "categoricas": categoricas,
            "discretas": discretas,
            "continuas": continuas,
            "continuas_yeo_johnson": continuas_sesgadas,
        },
        "SPLIT": {"test_size": TEST_SIZE, "random_state": RANDOM_STATE},
        "PREPROCESSING": {
            "imputacion_categoricas": "frequent",
            "imputacion_numericas": "median",
            "agrupacion_raras": {"tol": TOL_RARAS, "replace_with": "Rare"},
            "codificacion": {"metodo": "ordinal_ordered", "no_vistas": -1},
            "outliers": {"capping_method": "iqr", "tail": "both", "fold": FOLD_IQR},
            "transformacion": {"metodo": "yeo_johnson", "umbral_sesgo": UMBRAL_SESGO},
            "escalado": "standard",
        },
        "OUTPUTS": {
            **{
                nombre: ruta_relativa(PROCESSED_DATA_DIR / f"{nombre}.csv")
                for nombre in salidas
            },
            "pipeline": ruta_relativa(ruta_pipeline),
        },
    }
)

with open(CONFIG_FILE, "w", encoding="utf-8") as archivo:
    yaml.safe_dump(config, archivo, sort_keys=False, allow_unicode=True)

print(f"Pipeline guardado en: {ruta_pipeline.resolve()}")
print(f"Configuración guardada en: {CONFIG_FILE.resolve()}")